# Análise Descritiva - Capítulo 44

Este notebook contém uma análise descritiva sobre os dados de itens de nota fiscal do Capítulo 44 do NCM, conforme dados disponibilizados em ```tb_treina_cap44.csv```

## 1. Configuração Inicial e Carregamento de Dados (Pandas)

In [ ]:
import pandas as pd
import re
import os
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from IPython.display import display
from gensim.models import Word2Vec, FastText

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

FILE_PATH = 'outros_para_analise.csv'

# O arquivo é separado por tabulação ('\t')
try:
    df_pandas = pd.read_csv(FILE_PATH, sep='\t', decimal='.')
    print(f"Total de linhas: {len(df_pandas)}")
    
    df_pandas['prod_vuntrib'] = pd.to_numeric(df_pandas['prod_vuntrib'], errors='coerce')
    df_pandas['prod_qtrib'] = pd.to_numeric(df_pandas['prod_qtrib'], errors='coerce')
    df_pandas['contagem'] = pd.to_numeric(df_pandas['contagem'], errors='coerce', downcast='integer')
    
    df_pandas.info()
    print("\nPrimeiras 5 linhas:")
    print(df_pandas.head())
except Exception as e:
    print(f"Ocorreu um erro durante o carregamento do Pandas: {e}")

In [ ]:
# Normalização para MAIÚSCULAS
df_pandas['prod_xprod_norm'] = df_pandas['prod_xprod'].astype(str).str.upper()
print(df_pandas['prod_xprod_norm'].head())

## 2. Análise de Contagem (`contagem`)

In [ ]:
# Ordenar os produtos por contagem (qtd) (tanto asc como desc)
print("Top 50 Produtos por Contagem (Mais Frequentes)")
top_50_contagem = df_pandas.sort_values(by='contagem', ascending=False).head(50)
print(top_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

In [ ]:
print("\nBottom 50 Produtos por Contagem (Menos Frequentes)")
bottom_50_contagem = df_pandas.sort_values(by='contagem', ascending=True).head(50)
print(bottom_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

## 3. Análise de Preço (`prod_vuntrib` e `prod_qtrib`)

In [ ]:
print("Top 50 Produtos por Valor Unitário Tributável (prod_vuntrib)")
top_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=False).head(50)
print(top_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Valor Unitário Tributável (prod_vuntrib)")
bottom_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=True).head(50)
print(bottom_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Top 50 Produtos por Quantidade Tributável (prod_qtrib)")
top_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=False).head(50)
print(top_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Quantidade Tributável (prod_qtrib)")
bottom_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=True).head(50)
print(bottom_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

## 4. Análise de Representatividade (Valor Total)

In [ ]:
# Criar colunas de representatividade (Valor Total = Valor Unitário * Contagem)
df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['contagem']
df_pandas['valor_total_qtrib'] = df_pandas['prod_qtrib'] * df_pandas['contagem']

In [ ]:
print("Top 50 Produtos por Valor Total Tributável (valor_total_vuntrib)")
top_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=False).head(50)
print(top_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Valor Total Tributável (valor_total_vuntrib)")
bottom_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=True).head(50)
print(bottom_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

In [ ]:
print("Top 50 Produtos por Quantidade Total Tributável (valor_total_qtrib)")
top_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=False).head(50)
print(top_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Quantidade Total Tributável (valor_total_qtrib)")
bottom_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=True).head(50)
print(bottom_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

## Regex

In [ ]:
import pandas as pd
import re
import os

# ==============================================================================
# 1. DICIONÁRIOS DE TERMOS E EXCLUSÕES (INTEGRAÇÃO TOTAL)
# ==============================================================================

EXCL_MATERIAIS = r'ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|PLASTICO|ACRILICO|CERAMICA|PORCELANA|CONCRETO|CIMENTO|GRANITO|MARMORE|PEDRA'
EXCL_MOVEIS = r'ARMARIO|ESTANTE|MESA|CADEIRA|SOFA|POLTRONA|RACK|CRIADO MUDO|CAMA|BERCO|GUARDA ROUPA|COMODA|APARADOR|BUFFET|CRISTALEIRA|COZINHA|PLANEJADA|PLANEJADOS'
EXCL_DECORACAO = r'BRINQUEDO|BONECA|MINIATURA|ARTESANATO|ENFEITE|JOIA|BIJUTERIA|APLIQUE|RECORTE|LASER'
EXCL_SERVICOS = r'LOCACAO|ALUGUEL|REFORMA|CONSERTO|MANUTENCAO|REPARO|FRETE|MONTAGEM|INSTALACAO|TAXA|ENTREGA|AJUSTE|BALDEIO|USOS'

TERMOS_ESPECIES = r'CAMBARA|ANGELIM|TAUARI|CEDRINHO|CURUPIXA|ITAUBA|TATAJUBA|CUMARU|TECA|ABIURANA|CUPIUBA|AMESCLA|PINUS|EUCALIPTO|ARAUCO|GREENPLAC|CEDRO|MOGNO|VIROLA|IPÊ|IPE|JATOBÁ|JATOBA|SUCUPIRA|ASTRONIUN|ASTRONIUM|CLARISIA|RACEMOSA|URUNDEUVA|GUARIUBA|GUARIÚBA|ANGELIN|JEQUITIBA|CANELA|GARAPEIRA|GRANDIS|GIANDUIA|PINHAL|MAD\. LEI'
MADEIRA_SIMILARES = r'MAD|MAD\.|MADEIRA|MADEIRAS|MADEIR|MADERIA|MADEIRAW|BMADEIRA|MADEIRA1|MADEIRA3|MADEIRA_M3|EMADEIRA|MMADEIRA|MADEIRRA|MADDEIRA|DEMADEIRA|MADEIRAD|MADEIRADE|MADEIDRA|MADIEIRA|MADEIRAREFORÇADA|MADERIRA'
CARVAO_SIMILARES = r'CARVÃO|CARVAO|CARVOSUL|CARVOBOXX|CARVA|CARV|CAR|CACAO|CARVAOOLIVEIRA5KG|CARVAODOURADOS5KG|CARVOES|KICARVAO|CARS|CAVAO|CAVÃO'
MDF_SIMILARES = r'MDF|MDF4|MDF2|MDFC|MDFF|MDF05|MDF03|MDF02|MDF23173|MDF23174|MDF10|MDF23157|MDF15|NDF|HFDF|DF|MDX|MDFCRU|MD4|MDS|MD5|MDN|MD3|MDR|MDC|MD8|MDFREPARO|MSDF|MDCN'

# ==============================================================================
# 2. CONSOLIDAÇÃO DO DICIONÁRIO DE EXAUSTÃO (REDE DE SEGURANÇA)
# ==============================================================================
LISTA_EXAUSTAO = [
    TERMOS_ESPECIES, MADEIRA_SIMILARES, CARVAO_SIMILARES, MDF_SIMILARES,
    r'LENHA|TORA|ACHA|ESTILHA|SERRAGEM|RESIDUO|CAVACO|BIOMASSA|FINOS|OVERSIZE|MARAVALHA|GRANULADO|TORETE|CASCA',
    r'ARCO|ESTACA|MOIRAO|POSTE|DORMENTE|TABUA|PRANCHA|VIGA|VIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE',
    r'MOLDURA|RODAPE|TACO|LAMBRI|RODAMEIO|ALIZAR|OSB|MDP|PARTICULA|AGLOMERADO|COMPENSADO|TAPUME',
    r'PORTA|JANELA|BATENTE|CAIXILHO|FORRO|SOALHO|PISO|ASSOALHO|CABIDE|PALITO|CEPO|CARRETEL|BOBINA|URNA|CERCA|BANDEJA|PETISQUEIRA'
]
REGEX_EXAUSTAO = rf'\b(?:{"|".join(LISTA_EXAUSTAO)})\b'

# ==============================================================================
# 3. ESTRUTURA DE CATEGORIAS COMPLETA (NCM 44.01 A 44.21)
# ==============================================================================
categorias_regex = [
    # A. BLOQUEIO DE OUTLIERS
    ('OUTLIER__FINANCEIRO', rf'.*\b(?:{EXCL_SERVICOS}|LOTE MISTO|MERCADORIA MISTA)\b.*'),
    ('OUTLIER__MAQUINAS', r'.*\b(?:PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|GUINDASTE|EIXO|VW|DRILL|JOHN|DEERE)\b.*'),
    ('OUTLIER_OUTROS_CAPS', r'.*\b(?:BOVINOS)\b.*'),
    
    # B. ROTULADOS ANTES
    ('ROTULADOS ANTES', rf'.*\b(?:LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|(?:{CARVAO_SIMILARES}) VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)\b.*'),

    # C. NCM 44.01 a 44.06 (Matéria-Prima e Resíduos)
    ('44.01.LENHA_RESIDUOS', rf'^(?=.*(\b(?:LENHA|TORA|ACHA|ESTILHA|SERRAGEM|RESIDUO|CAVACO|BIOMASSA|FINOS|OVERSIZE|MARAVALHA|GRANULADO|TORETE|CASCA)\b))(?!.*(\b(?:{EXCL_MATERIAIS})\b)).*'),
    ('44.02.CARVAO_VEGETAL', rf'^(?=.*(?:\b(?:{CARVAO_SIMILARES})\b|\bMOINHA\b))(?!.*(\b(?:ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE)\b)).*'),
    ('44.03.MADEIRA_BRUTA', rf'^(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b.*\b(?:BRUTA|EM BRUTO)\b|\b(?:TORA|TORO|RONCO)\b))(?!.*(\b(?:SERRADA|APARELHADA|BENEFICIADA)\b)).*'),
    ('44.04.ARCOS_ESTACAS', rf'^(?=.*(\b(?:ARCO|ESTACA|MOIRAO|POSTE)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),
    ('44.05.LA_FARINHA', rf'^(?=.*(\b(?:LA DE MADEIRA|FARINHA DE MADEIRA|FIBRA DE MADEIRA)\b))(?!.*(\b(?:MINERAL|VIDRO|SINTETICA)\b)).*'),
    ('44.06.DORMENTES', rf'^(?=.*(\b(?:DORMENTE)\b))(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b|\b(?:FERROVIA|TRILHO)\b))(?!.*(\b(?:CONCRETO|METAL)\b)).*'),

    # D. NCM 44.07 a 44.09 (Madeira Processada)
    ('44.07.MADEIRA_SERRADA', rf'^(?=.*(?:\b(?:TABUA|PRANCHA|VIGA|VIGOTA|PRANCHÃO|PRANCHAO|PRANCHAS|MADEIRASERRADA|SERRADO|MADEIRA SERRADA|MADEIRA BENEFICIADA|BENEFICIADA|APARELHADA|SERRADA|MADEIRAVIGOTA|CAIBRO|RIPA|SARRAFO|BARROTE|PONTALETE)\b|\b(?:{TERMOS_ESPECIES})\b))(?!.*(\b(?:COMPENSADO|MDF|PAINEL)\b)).*'),
    ('44.08.FOLHAS_FOLHEADOS', rf'^(?=.*(\b(?:FOLHA PARA FOLHEADO|FOLHEADO|LAMINA|DESENROLADA)\b))(?=.*(\b(?:6MM|6 MM|FINA|DELGADA)\b)).*'),
    ('44.09.PERFILADOS', rf'^(?=.*(\b(?:MOLDURA|RODAPE|TACO|LAMBRI|RODAMEIO|ALIZAR)\b))(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b|\b(?:PINUS|EUCALIPTO)\b)).*'),

    # E. NCM 44.10 a 44.12 (Painéis e Chapas)
    ('44.10.PAINEIS_PARTICULAS_OSB', rf'^(?!.*\b(?:{EXCL_MOVEIS})\b)(?=.*(\b(?:OSB|MDP|PARTICULA|AGLOMERADO)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),
    ('44.11.PAINEIS_FIBRAS_MDF', rf'^(?!.*\b(?:{EXCL_MOVEIS})\b)(?=.*(?:\b(?:{MDF_SIMILARES})\b|\b(?:FIBRA|ARAUCO|GREENPLAC|CHAPATEX|DURATEX|BERNECK|EUCATEX)\b))(?=.*(\b(?:18MM|15MM|3MM|2F|1A)\b|\b(?:{MADEIRA_SIMILARES})\b)).*'),
    ('44.12.COMPENSADO', rf'^(?!.*\b(?:{EXCL_MOVEIS})\b)(?=.*(\b(?:COMPENSADO|CONTRAPLACADO|LAMINADO|NAVAL|SARRAFEADO|TAPUME)\b))(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b|\b(?:PINUS|EUCALIPTO)\b)).*'),

    # F. NCM 44.13 a 44.17 (Obras Diversas)
    ('44.13.MADEIRA_DENSIFICADA', rf'^(?=.*(\b(?:DENSIFICADA|ESTABILIZADA)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),
    ('44.14.MOLDURAS_QUADROS', rf'^(?=.*(\b(?:MOLDURA)\b))(?=.*(\b(?:QUADRO|ESPELHO|FOTO)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),
    ('44.15.CAIXOTES_PALETES', rf'^(?=.*(\b(?:PALETE|PALLET|ESTRADO|CAIXOTE|ENGRADADO|ESTRADO)\b))(?!.*(\b(?:{EXCL_SERVICOS})\b)).*'),
    ('44.16.BARRIS_TANOEIRO', rf'^(?=.*(\b(?:BARRIL|TONEL|ADUELA|CUBA|DORNA)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),
    ('44.17.FERRAMENTAS_CABOS', rf'^(?=.*(\b(?:FERRAMENTA|CABO|PÁ|ENXADA|VASSOURA)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),

    # G. NCM 44.18 (Marcenaria Construção)
    ('44.18.10.JANELAS', rf'^(?!.*(\b(?:{EXCL_MATERIAIS}|{EXCL_DECORACAO})\b))(?=.*(\b(?:JANELA|VENEZIANA|PORTA JANELA)\b))(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b|\b(?:{TERMOS_ESPECIES})\b)).*'),
    ('44.18.20.PORTAS', rf'^(?!.*(\b(?:{EXCL_MATERIAIS}|{EXCL_DECORACAO})\b))(?=.*(\b(?:PORTA|FOLHA.*PORTA|BATENTE|CAIXILHO)\b))(?!.*(\b(?:JANELA)\b)).*'),
    ('44.18.40.COFRAGEM', rf'^(?=.*(\b(?:COFRAGEM|FORMA.*CONCRETO)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),
    ('44.18.7X.PISOS_ASSOALHOS', rf'^(?=.*(\b(?:PISO|ASSOALHO|DECK|SOALHO|TACO)\b))(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b|\b(?:BAMBU)\b|\b(?:{TERMOS_ESPECIES})\b)).*'),
    ('44.18.9X.OUTROS_CONSTRUCAO', rf'^(?=.*(\b(?:VIGA|TRELIÇA|FORRO)\b))(?=.*(\b(?:CONSTRUCAO|OBRA)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),

    # H. NCM 44.19 a 44.21 (Artefatos e Utensílios)
    ('44.19.UTENSILIOS_MESA', rf'^(?=.*(\b(?:PRATO|COPO|TIGELA|COLHER|BANDEJA|TABUA.*CARNE)\b))(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b|\b(?:BAMBU)\b)).*'),
    ('44.20.MARCHETARIA_ESTATUETAS', rf'^(?!.*(\b(?:{EXCL_DECORACAO})\b))(?=.*(\b(?:MARCHETARIA|ESTATUETA|COFRE|CAIXA.*JOIA)\b))(?=.*\b(?:{MADEIRA_SIMILARES})\b).*'),
    ('44.21.OUTRAS_OBRAS', rf'^(?!.*(\b(?:{EXCL_MATERIAIS})\b))(?=.*(\b(?:CABIDE|PALITO|CEPO|CARRETEL|BOBINA|URNA|CERCA|BANDEJA|PETISQUEIRA|TIGELA|CUMBUCA|ESTATUETA|CAIXA|PRENDEDOR|RETRATO)\b))(?=.*(?:\b(?:{MADEIRA_SIMILARES})\b|\b(?:BAMBU)\b)).*'),

    # I. CATEGORIA DE REVISÃO E FECHAMENTO
    ('AMBIGUOS_REVISAR', rf'.*\b(?:PLANEJADA|COZINHA|DALMOBILE|LOTE MISTO|MERCADORIA MISTA)\b.*'),
    ('OUTROS', r'.*')
]

# ==============================================================================
# 4. FUNÇÃO DE CATEGORIZAÇÃO COM REDE DE SEGURANÇA
# ==============================================================================
def categorizar_produto_final(descricao):
    descricao_upper = str(descricao).upper()
    
    # 1. Tenta classificar nas regras específicas
    for categoria, regex in categorias_regex:
        if categoria == 'OUTROS':
            break
        if re.search(regex, descricao_upper):
            return categoria
            
    # 2. REDE DE SEGURANÇA POR EXAUSTÃO
    if re.search(REGEX_EXAUSTAO, descricao_upper):
        return 'ELIMINADOS_POR_REGRA'
        
    # 3. Se não tem NENHUMA palavra do dicionário, é verdadeiramente OUTROS
    return 'OUTROS'

# ==============================================================================
# 5. EXECUÇÃO E FILTRAGEM
# ==============================================================================
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto_final)

# Separação das bases para conferência final
categorias_cap_44 = [c[0] for c in categorias_regex if c[0].startswith('44.') or c[0] == 'ROTULADOS ANTES']
categorias_eliminadas = ['OUTLIER__FINANCEIRO', 'OUTLIER__MAQUINAS', 'OUTLIER_OUTROS_CAPS', 'AMBIGUOS_REVISAR', 'ELIMINADOS_POR_REGRA']

df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

# Exibição dos resultados
print(f"1. CAPÍTULO 44 (Granular): {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:                 {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS:             {len(df_eliminados):>10,}".replace(",", "."))


Resultado Atual

1. CAPÍTULO 44 (Granular):    819.493
2. OUTROS:                    923.425
3. ELIMINADOS:                 12.457

A versão do Copilot/IA eu preciso analisar com mais detalhes, mas tem potencial.

In [ ]:
print(" Distribuição de Produtos por Categoria Regex (Top 20)")
distribuicao_regex = df_pandas['categoria_regex'].value_counts().head(20)
print(distribuicao_regex)

print(" Representatividade Financeira por Categoria (Top 20)")
representatividade_regex = (
    df_pandas.groupby('categoria_regex')['valor_total_vuntrib']
    .sum()
    .sort_values(ascending=False)
)

total = representatividade_regex.sum()
representatividade_percentual = (representatividade_regex / total) * 100
print(representatividade_percentual.head(20).round(2).astype(str) +"%")

In [ ]:
# 1. Representatividade por CONTAGEM (Volume de Dados)
contagem_absoluta = df_pandas['categoria_regex'].value_counts()
total_linhas = len(df_pandas)

representatividade_contagem_pct = (contagem_absoluta / total_linhas) * 100
print(representatividade_contagem_pct.head(20).round(2).astype(str) +"%")

# 2. Comparação Direta (Contagem vs Financeiro) 
comparativo = pd.DataFrame({
    'Volume (%)': representatividade_contagem_pct,
    'Financeiro (%)': representatividade_percentual
})
print(comparativo.head(20).round(2))


In [ ]:
def gerar_referencial_percentual(df):
    # 1. Cálculos de Contagem (Volume de Linhas)
    contagem_absoluta = df['categoria_regex'].value_counts()
    total_linhas = len(df)
    contagem_percentual = (contagem_absoluta / total_linhas) * 100

    # 2. Cálculos de Valor Financeiro (Impacto Econômico)
    valor_absoluto = df.groupby('categoria_regex')['valor_total_vuntrib'].sum()
    total_valor = valor_absoluto.sum()
    valor_percentual = (valor_absoluto / total_valor) * 100

    # 3. Consolidação em um único DataFrame
    df_analise = pd.DataFrame({
        'Contagem_Absoluta': contagem_absoluta,
        'Contagem_Percentual': contagem_percentual,
        'Valor_Total_Absoluto': valor_absoluto,
        'Valor_Percentual': valor_percentual
    }).sort_values(by='Valor_Percentual', ascending=False)

    # 4. Formatação para exibição
    df_display = df_analise.copy()
    df_display['Contagem_Percentual'] = df_display['Contagem_Percentual'].map('{:.2f}%'.format)
    df_display['Valor_Percentual'] = df_display['Valor_Percentual'].map('{:.2f}%'.format)
    df_display['Valor_Total_Absoluto'] = df_display['Valor_Total_Absoluto'].map('R$ {:,.2f}'.format)
    df_display['Contagem_Absoluta'] = df_display['Contagem_Absoluta'].map('{:,}'.format)

    print("\n" + "="*80)
    print("REFERENCIAL PERCENTUAL COMPLETO (TODAS AS CATEGORIAS)")
    print("="*80)
    display(df_display) # Use display() para uma tabela bonita no Jupyter
    print("="*80)
    
    # Resumo por Grupos Estratégicos
    print("\nRESUMO POR GRUPOS ESTRATÉGICOS:")
    categorias_cap44 = [c for c in df_analise.index if c.startswith('44.') or c == 'ROTULADOS ANTES']
    categorias_outliers = [c for c in df_analise.index if 'OUTLIER' in c or c == 'AMBIGUOS_REVISAR']
    
    resumo = {
        'CAPÍTULO 44': df_analise.loc[df_analise.index.isin(categorias_cap44)].sum(),
        'OUTROS': df_analise.loc[df_analise.index == 'OUTROS'].sum(),
        'ELIMINADOS/OUTLIERS': df_analise.loc[df_analise.index.isin(categorias_outliers)].sum()
    }
    
    df_resumo = pd.DataFrame(resumo).T
    df_resumo['Contagem_Percentual'] = (df_resumo['Contagem_Absoluta'] / total_linhas * 100).map('{:.2f}%'.format)
    df_resumo['Valor_Percentual'] = (df_resumo['Valor_Total_Absoluto'] / total_valor * 100).map('{:.2f}%'.format)
    
    display(df_resumo[['Contagem_Absoluta', 'Contagem_Percentual', 'Valor_Total_Absoluto', 'Valor_Percentual']])
    print("="*80)

    return df_analise

# Execução
df_referencial = gerar_referencial_percentual(df_pandas)


## 6. Para análises específicas

In [ ]:
df_pandas.head()

In [ ]:
# 1. Definir as categorias que você deseja filtrar
categorias_carvao = ['ELIMINADOS_POR_REGRA']

# 2. Criar um novo DataFrame apenas com essas categorias
df_apenas_carvao = df_pandas[df_pandas['categoria_regex'].isin(categorias_carvao)].copy()

# 3. Salvar em um arquivo CSV específico
df_apenas_carvao.to_csv('eliminados_para_analise.csv', sep='\t', index=False, encoding='utf-8-sig')

# 4. Exibir a contagem para conferência
print(f"Total de itens eliminados por regra: {len(df_apenas_carvao)}")


In [ ]:
'''ITEM ="MDF"
base ="cap_44.csv"
mask = df_pandas.apply(lambda col: col.astype(str).str.contains(ITEM, case=False, na=False))
df_filtrado = df_pandas[mask.any(axis=1)]
arquivo_saida = f"{ITEM}.csv"
df_filtrado.to_csv(arquivo_saida, index=False)'''

In [ ]:
df_outros = df_pandas[df_pandas['categoria_regex'] =="OUTROS"]
arquivo_saida ="OUTROS.csv"
df_outros.to_csv(arquivo_saida, index=False)

In [ ]:
df_validacao = df_pandas[['prod_xprod', 'categoria_regex']].copy()

# Usamos sep='\t'
df_validacao.to_csv('validacao_categorias_ncm44.csv', sep='\t', index=False, encoding='utf-8-sig')

for cat in df_validacao['categoria_regex'].unique():
    print(f" Amostra da Categoria: {cat}")
    print(df_validacao[df_validacao['categoria_regex'] == cat]['prod_xprod'].head(3).tolist())

 
## 7. Novas Ferramentas de NLP (BoW, TF-IDF e Embeddings)

### 2. Bag of Words (BoW)
Representação de contagem de palavras para o dataset principal e filtros específicos.

In [ ]:
# 2.1 Bag of Words com Visualização
vectorizer_bow = CountVectorizer(max_features=1000, stop_words=['DE', 'DA', 'DO', 'COM', 'PARA', 'EM'])
bow_matrix = vectorizer_bow.fit_transform(df_pandas['prod_xprod_norm'])

sum_words = bow_matrix.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer_bow.vocabulary_.items()]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

df_bow_viz = pd.DataFrame(words_freq, columns=['Termo', 'Frequência'])
print("\nTop 30 Termos - Bag of Words:")
display(df_bow_viz.head(30))

### 3. TF-IDF
Representação de frequência inversa de documentos para o dataset principal e filtros específicos.

In [ ]:
# 3.1 TF-IDF com Visualização
print("Processando TF-IDF...")

vectorizer_tfidf = TfidfVectorizer(
    max_features=1000, 
    max_df=0.7,  # IGNORE termos que aparecem em mais de 70% dos itens
    min_df=5,    # IGNORE termos que aparecem em menos de 5 itens
    ngram_range=(1, 2) # Captura"MADEIRA PINUS"como um único termo
)
tfidf_matrix = vectorizer_tfidf.fit_transform(df_pandas['prod_xprod_norm'])

# Criando um DataFrame para visualizar os Top 30 termos por Score Médio
feature_names = vectorizer_tfidf.get_feature_names_out()
avg_tfidf = tfidf_matrix.mean(axis=0).tolist()[0]
tfidf_scores = sorted(list(zip(feature_names, avg_tfidf)), key=lambda x: x[1], reverse=True)

df_tfidf_viz = pd.DataFrame(tfidf_scores, columns=['Termo', 'Score_TFIDF'])
print("\nTop 30 Termos - TF-IDF (Importância Relativa):")
display(df_tfidf_viz.head(30))

### 4. Word Embeddings
Implementação de Word2Vec e FastText.

In [ ]:
def simple_tokenize(text):
    return re.findall(r'\w+', str(text).upper())

sentences = df_pandas['prod_xprod_norm'].apply(simple_tokenize).tolist()
print(f"Tokenização: {len(sentences)} sentenças preparadas.")

# 4.1 Word2Vec
print("Treinando Word2Vec")
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4)

termo_teste = 'MDF'
if termo_teste in w2v_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste}':")
    display(pd.DataFrame(w2v_model.wv.most_similar(termo_teste, topn=20), columns=['Termo', 'Similaridade']))

In [ ]:
# 4.2 FastText - Treinamento
print("Treinando FastText (lidando com sub-palavras e erros de digitação)...")
ft_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4)
print("FastText treinado com sucesso!")

termo_teste_ft = 'CARVÃO'
if termo_teste_ft in ft_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste_ft}' (via FastText):")
    display(pd.DataFrame(ft_model.wv.most_similar(termo_teste_ft, topn=30), columns=['Termo', 'Similaridade']))

In [ ]:
# 4.2 FastText - Treinamento
print("Treinando FastText (lidando com sub-palavras e erros de digitação)...")
ft_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4)
print("FastText treinado com sucesso!")

termo_teste_ft = 'MDF'
if termo_teste_ft in ft_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste_ft}' (via FastText):")
    display(pd.DataFrame(ft_model.wv.most_similar(termo_teste_ft, topn=30), columns=['Termo', 'Similaridade']))

## Exportação de dados financeiros para análise

In [ ]:
df = df_pandas

# 2. NORMALIZAÇÃO E LIMPEZA
df['prod_xprod_norm'] = df['prod_xprod'].astype(str).str.upper()
df['prod_vuntrib'] = pd.to_numeric(df['prod_vuntrib'], errors='coerce').fillna(0)
df['prod_qtrib'] = pd.to_numeric(df['prod_qtrib'], errors='coerce').fillna(0)
df['contagem'] = pd.to_numeric(df['contagem'], errors='coerce').fillna(1)

# Calcular valor total por linha (Valor Unitário * Quantidade de Ocorrências)
df['valor_total_vuntrib'] = df['prod_vuntrib'] * df['contagem']
df['valor_total_qtrib'] = df['prod_qtrib'] * df['contagem']

# Stopwords a ignorar (Palavras irrelevantes para a análise financeira)
stopwords = {
    'DE', 'DA', 'DO', 'COM', 'PARA', 'EM', 'UM', 'UMA', 'CM', 'MM', 'MT', 
    'KG', 'UN', 'UND', 'C/', 'X', 'O', 'A', 'OS', 'AS', 'PARA', 'POR', 'NAS'
}

# Dicionário para acumular estatísticas: {palavra: [soma_vuntrib, soma_qtrib, contagem_ocorrencias]}
word_stats = defaultdict(lambda: [0.0, 0.0, 0])

# 3. PROCESSAMENTO POR LINHA
for idx, row in df.iterrows():
    words = set(re.findall(r'\w+', row['prod_xprod_norm']))
    v_total = row['valor_total_vuntrib']
    q_total = row['valor_total_qtrib']
    
    for word in words:
        # Filtra stopwords, números puros e palavras muito curtas
        if word not in stopwords and not word.isdigit() and len(word) > 1:
            stats = word_stats[word]
            stats[0] += v_total
            stats[1] += q_total
            stats[2] += 1

# 4. CONSOLIDAÇÃO DOS RESULTADOS
result_data = []
for word, stats in word_stats.items():
    result_data.append({
        'Termo': word,
        'Valor_Total_Vuntrib': stats[0],
        'Valor_Total_Qtrib': stats[1],
        'Ocorrencias': stats[2],
        'Valor_Medio_Vuntrib': stats[0] / stats[2] if stats[2] > 0 else 0,
        'Valor_Medio_Qtrib': stats[1] / stats[2] if stats[2] > 0 else 0
    })

df_results = pd.DataFrame(result_data)

top_50_total_vuntrib = df_results.sort_values(by='Valor_Total_Vuntrib', ascending=False).head(50)
top_50_medio_vuntrib = df_results[df_results['Ocorrencias'] > 10].sort_values(by='Valor_Medio_Vuntrib', ascending=False).head(50)
top_50_total_qtrib = df_results.sort_values(by='Valor_Total_Qtrib', ascending=False).head(50)
top_50_medio_qtrib = df_results[df_results['Ocorrencias'] > 10].sort_values(by='Valor_Medio_Qtrib', ascending=False).head(50)

top_50_total_vuntrib.to_csv('top_50_total_vuntrib.csv', index=False)
top_50_medio_vuntrib.to_csv('top_50_medio_vuntrib.csv', index=False)
top_50_total_qtrib.to_csv('top_50_total_qtrib.csv', index=False)
top_50_medio_qtrib.to_csv('top_50_medio_qtrib.csv', index=False)



In [ ]:
def analisar_valor_transacao_por_termo(df):
    
    df['valor_transacao_total'] = df['prod_vuntrib'] * df['prod_qtrib'] * df['contagem']

    # Stopwords a ignorar
    stopwords = {
        'DE', 'DA', 'DO', 'COM', 'PARA', 'EM', 'UM', 'UMA', 'CM', 'MM', 'MT', 
        'KG', 'UN', 'UND', 'C/', 'X', 'O', 'A', 'OS', 'AS', 'POR', 'NAS', 'NOS',
        'E', 'OU', 'SE', 'NA', 'NO', 'NOS', 'NAS', 'AO', 'AOS', 'AS', 'ESTE', 'ESTA',
        'ESTES', 'ESTAS', 'ESSE', 'ESSA', 'ESSES', 'ESSAS', 'AQUELA', 'AQUELE', 'AQUELAS', 'AQUELOS'
    }

    word_stats = defaultdict(lambda: [0.0, 0])
    
    def processar_linha(row):
        words = set(re.findall(r'\w+', row['prod_xprod_norm']))
        v_transacao = row['valor_transacao_total']
        
        for word in words:
            # Filtros: Stopwords, dígitos e tamanho > 1
            if word not in stopwords and not word.isdigit() and len(word) > 1:
                stats = word_stats[word]
                stats[0] += v_transacao
                stats[1] += 1

    df.apply(processar_linha, axis=1)

    # 2. Consolidação
    result_data = []
    for word, stats in word_stats.items():
        result_data.append({
            'Termo': word,
            'Valor_Transacao_Total': stats[0],
            'Ocorrencias': stats[1],
            'Valor_Transacao_Medio': stats[0] / stats[1] if stats[1] > 0 else 0
        })

    df_results = pd.DataFrame(result_data)

    
    # Top 100 Total
    top_100_total = df_results.sort_values(by='Valor_Transacao_Total', ascending=False).head(100)
    top_100_total.to_csv('top_100_total_transacao_2.csv', index=False, encoding='utf-8-sig')
    
    # Top 100 Médio
    top_100_medio = df_results[df_results['Ocorrencias'] >= 10].sort_values(by='Valor_Transacao_Medio', ascending=False).head(100)
    top_100_medio.to_csv('top_100_medio_transacao_2.csv', index=False, encoding='utf-8-sig')

    print(f"Total de termos únicos: {len(df_results)}")

    #df_results.to_csv('termos_unicos_outros.csv', index=False)
    
    return df_results


In [ ]:
if 'df_pandas' in locals() and 'prod_xprod_norm' in df_pandas.columns:
    df_termos_analisados = analisar_valor_transacao_por_termo(df_pandas)
else:
    print("Erro: O DataFrame 'df_pandas' ou a coluna 'prod_xprod_norm' não foram encontrados. Certifique-se de executar as células anteriores.")

In [ ]:
df_results.to_csv('termos_unicos.csv', index=False)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

FILE_PATH = 'eliminados_para_analise.csv'

# O arquivo é separado por tabulação ('\t')
try:
    df_outros = pd.read_csv(FILE_PATH, sep=';', decimal='.')
    print(f"Total de linhas: {len(df_outros)}")
    
    df_outros['prod_vuntrib'] = pd.to_numeric(df_outros['prod_vuntrib'], errors='coerce')
    df_outros['prod_qtrib'] = pd.to_numeric(df_outros['prod_qtrib'], errors='coerce')
    df_outros['contagem'] = pd.to_numeric(df_outros['contagem'], errors='coerce', downcast='integer')
    
    df_outros.info()
    print("\nPrimeiras 5 linhas:")
    print(df_outros.head())
except Exception as e:
    print(f"Ocorreu um erro durante o carregamento do Pandas: {e}")

In [ ]:
# Normalização para MAIÚSCULAS
df_outros['prod_xprod_norm'] = df_outros['prod_xprod'].astype(str).str.upper()
print(df_outros['prod_xprod_norm'].head())

In [ ]:
if 'df_outros' in locals() and 'prod_xprod_norm' in df_outros.columns:
    df_termos_analisados = analisar_valor_transacao_por_termo(df_outros)
else:
    print("Erro: O DataFrame 'df_outros' ou a coluna 'prod_xprod_norm' não foram encontrados. Certifique-se de executar as células anteriores.")

In [ ]:
# gerar lista única de descrições
df_unicos = (
    df_outros['prod_xprod']
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
    .to_frame()
)

# salvar com separador ;
df_unicos.to_csv(
    'eliminados_para_analise_unicos.csv',
    sep=';',
    index=False,
    encoding='utf-8-sig'
)

print(f"Total de descrições únicas: {len(df_unicos):,}".replace(",", "."))


## Análise - OUTROS restantes

De 879.846 para 301.695